In [ ]:
✔ all()
Returns all records.
Get all objects.
      User.objects.all()
    
<QuerySet [<User: user1>, <User: user2>, ...]>



In [ ]:
✔ filter()

Filter by conditions.
      User.objects.filter(is_active=True)
    
<QuerySet [<User: user1>, <User: user3>]>


In [ ]:
✔ exclude()

Opposite of filter.
     User.objects.exclude(is_staff=True)
<QuerySet [<User: user2>]>


In [ ]:
✔ get()

Get a single object (must return EXACTLY ONE).
     User.objects.get(email="a@test.com")
<User: user1>


In [ ]:
✔ first(), last()
   User.objects.filter(role="admin").first()
    User.objects.last()


In [ ]:
✔ count()
    User.objects.filter(active=True).count() 
10
✔ exists()
   User.objects.filter(email="a@test.com").exists()
True

In [ ]:
⭐ 2. Ordering + Limiting
✔ order_by()

User.objects.order_by("name")
User.objects.order_by("-created_at")

<QuerySet ordered by username ascending>


In [ ]:
✔ reverse()
    User.objects.order_by("name").reverse()

id in descending order

✔ distinct()
   User.objects.values("role").distinct()


In [ ]:
✅ 6. .values()
User.objects.values("id", "username")
<QuerySet [{'id': 1, 'username': 'user1'}, {'id': 2, 'username': 'user2'}]>


In [ ]:
✅ 7. .values_list()
User.objects.values_list("id", "email")

<QuerySet [(1, 'a@gmail.com'), (2, 'b@gmail.com')]>


In [ ]:
✔ Limit Results (via slicing)
User.objects.all()[:10]
User.objects.all()[20:40]


In [ ]:
⭐ 3. Field Lookup Types
User.objects.filter(name__icontains="john")
User.objects.filter(email__startswith="test")
User.objects.filter(role__endswith="admin")


In [ ]:
Numeric lookups:

Product.objects.filter(price__gt=100)
Product.objects.filter(price__lt=50)
Product.objects.filter(price__range=(10, 20))


In [ ]:
Date lookups:
Order.objects.filter(created_at__date="2025-01-01")
Order.objects.filter(created_at__year=2025)
Order.objects.filter(created_at__month=5)


In [ ]:
Null checks:
User.objects.filter(phone__isnull=True)


In [ ]:
⭐ 4. CRUD Methods
User.objects.create(name="John", email="john@test.com")


In [ ]:
2. Create object then save
user = User(name="John")
user.save()

In [ ]:
3. bulk_create()
User.objects.bulk_create([
    User(name="A"),
    User(name="B"),
])


In [ ]:
✔ UPDATE  1. Update single object
user = User.objects.get(id=1)
user.name = "Updated"
user.save()


In [ ]:
2. Update many rows at once
User.objects.filter(active=False).update(active=True)


In [ ]:
3. update_or_create()
User.objects.update_or_create(email="a@test.com", defaults={"name": "John"})


In [ ]:
4. get_or_create()
User.objects.get_or_create(email="a@test.com", defaults={"name": "John"})


In [ ]:
✔ DELETE
1. delete object
user = User.objects.get(id=1)
user.delete()

In [ ]:
2. delete multiple
User.objects.filter(active=False).delete()


In [ ]:
⭐ 5. Aggregations
from django.db.models import Sum, Count, Max, Min, Avg
User.objects.count()
Order.objects.aggregate(Sum("total"))
Order.objects.aggregate(Max("total"))
Product.objects.aggregate(Avg("price"))

{'total': 100}


In [ ]:
Group By → annotate()
User.objects.values("role").annotate(total=Count("id"))

<QuerySet [{'name':'Admin', 'total_users': 3}]>


In [ ]:
✅ 18. .select_related()

Foreign-key JOIN → improves performance.
Book.objects.select_related("author").all()
Single optimized SQL join.

In [ ]:
✅ 19. .prefetch_related()
For many-to-many and reverse relationships.
Author.objects.prefetch_related("books")
<QuerySet [Author with prefetched books]>


In [ ]:
✅ 20. .raw()

User.objects.raw("SELECT * FROM auth_user")
<RawQuerySet [User1, User2]>


In [ ]:
✅ 21. .distinct()
User.objects.distinct('email')
Unique email rows


In [ ]:
✅ 1. Add Multiple Databases in settings.py

# settings.py
DATABASES = {
    "default": {
        "ENGINE": "django.db.backends.postgresql",
        "NAME": "main_db",
        "USER": "postgres",
        "PASSWORD": "password",
        "HOST": "localhost",
        "PORT": "5432",
    },

    "analytics": {
        "ENGINE": "django.db.backends.mysql",
        "NAME": "analytics_db",
        "USER": "root",
        "PASSWORD": "root123",
        "HOST": "localhost",
        "PORT": "3306",
    }
}
✔ default → main application database
✔ analytics → second database (reports, logs, etc.)

In [ ]:
✅ 2. How to Use Specific Database in Queries

Model.objects.using("<database_name>")

User.objects.using("analytics").all()

User.objects.using("analytics").create(username="john")


In [ ]:
⭐ E. Raw SQL on a specific DB

User.objects.raw("SELECT * FROM users", using="analytics")


In [ ]:
⭐ F. Transactions on a specific DB

from django.db import transaction

with transaction.atomic(using="analytics"):
    User.objects.using("analytics").create(username="mark")

In [ ]:
“Django supports multiple databases using the DATABASES setting.
We can run queries on a particular DB using .using("<db_name>").
For real applications, we use a database router to automatically route
read/write operations based on the model's app.
Migrations are also controlled using the router’s allow_migrate() method.”

In [ ]:
📁 Create database_router.py

class AnalyticsRouter:
    """
    Route models in 'analytics' app to the 'analytics' DB
    """
    def db_for_read(self, model, **hints):
        if model._meta.app_label == "analytics":
            return "analytics"
        return None

    def db_for_write(self, model, **hints):
        if model._meta.app_label == "analytics":
            return "analytics"
        return None

    def allow_relation(self, obj1, obj2, **hints):
        if obj1._meta.app_label == "analytics" or obj2._meta.app_label == "analytics":
            return True
        return None

    def allow_migrate(self, db, app_label, model_name=None, **hints):
        if app_label == "analytics":
            return db == "analytics"
        return None


In [ ]:
📌 Add this router in settings.py

DATABASE_ROUTERS = ["project.database_router.AnalyticsRouter"]


In [ ]:
📁 project/db_router.py

class MasterSlaveRouter:

    def db_for_read(self, model, **hints):
        return "replica"

    def db_for_write(self, model, **hints):
        return "default"

    def allow_relation(self, obj1, obj2, **hints):
        return True

    def allow_migrate(self, db, app_label, model_name=None, **hints):
        # Run migrations only on MASTER
        return db == "default"
